<a href="https://colab.research.google.com/github/Ayushprajapati2006/AI-LAB-/blob/main/week%207.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import re

# Check whether a term is a variable
# Convention: lowercase names without arguments are variables
def is_variable(term):
    name, args = term
    return len(args) == 0 and name[0].islower()


# Apply substitutions to a term
def substitute(term, subs):
    name, args = term

    if is_variable(term) and name in subs:
        return substitute(subs[name], subs)

    return (name, tuple(substitute(arg, subs) for arg in args))


# Check whether a variable occurs inside a term
def occurs(variable, term, subs):
    term = substitute(term, subs)

    if is_variable(term):
        return term[0] == variable

    return any(occurs(variable, arg, subs)
               for arg in term[1])


# Convert input expression into a tree
def parse(expression):
    tokens = re.findall(r'[A-Za-z_]\w*|[(),]', expression)
    position = [0]

    def parse_term():
        if position[0] >= len(tokens):
            raise ValueError("Invalid expression")

        name = tokens[position[0]]
        position[0] += 1

        args = []

        if position[0] < len(tokens) and tokens[position[0]] == '(':
            position[0] += 1

            args.append(parse_term())

            while position[0] < len(tokens) and tokens[position[0]] == ',':
                position[0] += 1
                args.append(parse_term())

            if position[0] >= len(tokens) or tokens[position[0]] != ')':
                raise ValueError("Missing closing bracket")

            position[0] += 1

        return (name, tuple(args))

    result = parse_term()

    if position[0] != len(tokens):
        raise ValueError("Invalid expression")

    return result


# Convert a term back into readable expression
def display(term):
    name, args = term

    if not args:
        return name

    return name + "(" + ", ".join(display(a) for a in args) + ")"


# Main unification algorithm
def unify(term1, term2, subs):

    term1 = substitute(term1, subs)
    term2 = substitute(term2, subs)

    # Both terms are identical
    if term1 == term2:
        return True

    # First term is a variable
    if is_variable(term1):
        variable = term1[0]

        if occurs(variable, term2, subs):
            return False

        subs[variable] = term2
        return True

    # Second term is a variable
    if is_variable(term2):
        variable = term2[0]

        if occurs(variable, term1, subs):
            return False

        subs[variable] = term1
        return True

    # Check predicate/function names and argument counts
    if term1[0] != term2[0]:
        return False

    if len(term1[1]) != len(term2[1]):
        return False

    # Recursively unify corresponding arguments
    for a, b in zip(term1[1], term2[1]):
        if not unify(a, b, subs):
            return False

    return True


# Main program
def main():

    print("FIRST ORDER LOGIC - UNIFICATION")
    print("--------------------------------")

    exp1 = input("Enter first expression: ")
    exp2 = input("Enter second expression: ")

    try:
        term1 = parse(exp1)
        term2 = parse(exp2)

        substitutions = {}

        print("\nExpression 1:", display(term1))
        print("Expression 2:", display(term2))

        if unify(term1, term2, substitutions):

            print("\nUnification Successful!")

            print("\nMost General Unifier (MGU):")

            if substitutions:
                for variable in sorted(substitutions):
                    value = substitute(
                        substitutions[variable],
                        substitutions
                    )
                    print(variable, "=", display(value))
            else:
                print("No substitutions required.")

            print("\nUnified Expression:")
            print(display(substitute(term1, substitutions)))

        else:
            print("\nUnification Failed!")
            print("The expressions cannot be unified.")

    except (ValueError, IndexError):
        print("\nInvalid expression entered!")


if __name__ == "__main__":
    main()
print("Ayush Prajapati 1WA24CS075")

FIRST ORDER LOGIC - UNIFICATION
--------------------------------
Enter first expression: eats(x,apple)
Enter second expression: eats(riya,y)

Expression 1: eats(x, apple)
Expression 2: eats(riya, y)

Unification Successful!

Most General Unifier (MGU):
apple = y
x = riya

Unified Expression:
eats(riya, y)
Ayush Prajapati 1WA24CS075
